In [ ]:
# ── Colab Setup (skip automatically if running locally) ──────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # 1. Install NeuralForecast (PyTorch is pre-installed on Colab)
    import subprocess
    subprocess.run(['pip', 'install', 'neuralforecast', '-q'], check=True)

    # 2. Clone the repo to get scripts/models utility code
    REPO_URL = 'https://github.com/WoodyChang21/ECE1508_GenAI.git'
    if not os.path.exists('/content/ECE1508_GenAI'):
        subprocess.run(
            ['git', 'clone', '--branch', 'Model', REPO_URL, '/content/ECE1508_GenAI'],
            check=True,
        )
    os.chdir('/content/ECE1508_GenAI/notebooks')

    # 3. Upload data splits — upload train.parquet, val.parquet, test.parquet
    #    from your local  data/splits/  folder when prompted
    os.makedirs('/content/ECE1508_GenAI/data/splits', exist_ok=True)
    os.makedirs('/content/ECE1508_GenAI/data/predictions', exist_ok=True)

    from google.colab import files as colab_files
    print("Upload train.parquet, val.parquet, and test.parquet from your local data/splits/ folder:")
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        dest = f'/content/ECE1508_GenAI/data/splits/{fname}'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f"  Saved → {dest}")

    # 4. Verify GPU (strongly recommended — CPU training takes hours)
    import torch
    if torch.cuda.is_available():
        print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    else:
        print("\nNo GPU detected. Go to Runtime → Change runtime type → T4 GPU before running training cells.")

print("Setup complete.")

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import DeepAR
from neuralforecast.losses.pytorch import DistributionLoss, MQLoss

from scripts.models.data_loader import load_nf_dataframe, build_full_df, FUTR_EXOG_COLS
from scripts.models.metrics import compute_all

plt.rcParams['figure.figsize'] = (14, 4)

CANDIDATES = [24, 60, 120, 240]
FREQ = 1   # integer ds — avoids market-hours gap issues

In [ ]:
extra = FUTR_EXOG_COLS  # ['is_first_bar']

train_df = load_nf_dataframe('../data/splits/train.parquet', extra_cols=extra)
val_df   = load_nf_dataframe('../data/splits/val.parquet',   extra_cols=extra)
test_df  = load_nf_dataframe('../data/splits/test.parquet',  extra_cols=extra)

# Contiguous ds sequences for cross_validation
trainval_df = build_full_df([train_df, val_df])
full_df     = build_full_df([train_df, val_df, test_df])

print(f"train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")
print(f"trainval: {len(trainval_df):,}  full: {len(full_df):,}")
print(f"futr_exog columns: {FUTR_EXOG_COLS}")
print(full_df.head(3))

In [ ]:
val_maes = {}

for input_size in CANDIDATES:
    model = DeepAR(
        h=1,
        input_size=input_size,
        lstm_hidden_size=128,
        lstm_n_layers=2,
        lstm_dropout=0.1,
        trajectory_samples=100,          # fewer samples for speed during tuning
        loss=DistributionLoss(distribution='StudentT', level=[80, 90]),
        valid_loss=MQLoss(level=[80, 90]),
        futr_exog_list=FUTR_EXOG_COLS,
        max_steps=500,
        early_stop_patience_steps=-1,    # disable early stopping; cross_validation has no val split
        scaler_type='standard',
    )
    nf = NeuralForecast(models=[model], freq=FREQ)
    # cross_validation treats last test_size rows of trainval_df as the "test" (= val here)
    # n_windows=None is required when test_size is provided (NeuralForecast 1.7.7 API)
    cv = nf.cross_validation(
        df=trainval_df,
        n_windows=None,
        test_size=len(val_df),
        step_size=1,
        refit=False,
    )
    val_mae = float(np.mean(np.abs(cv['y'].values - cv['DeepAR'].values)))
    val_maes[input_size] = val_mae
    print(f"  input_size={input_size:3d}  val MAE={val_mae:.6f}")

best_input_size = min(val_maes, key=val_maes.get)
print(f"\nBest input_size: {best_input_size}  (val MAE={val_maes[best_input_size]:.6f})")

In [ ]:
# Train on train+val (full_df minus last test_size rows), predict across test
final_model = DeepAR(
    h=1,
    input_size=best_input_size,
    lstm_hidden_size=128,
    lstm_n_layers=2,
    lstm_dropout=0.1,
    trajectory_samples=200,
    loss=DistributionLoss(distribution='StudentT', level=[80, 90]),
    valid_loss=MQLoss(level=[80, 90]),
    futr_exog_list=FUTR_EXOG_COLS,
    max_steps=1000,
    early_stop_patience_steps=-1,    # disable early stopping; cross_validation has no val split
    scaler_type='standard',
)
nf_final = NeuralForecast(models=[final_model], freq=FREQ)

# n_windows=None is required when test_size is provided (NeuralForecast 1.7.7 API)
cv_test = nf_final.cross_validation(
    df=full_df,
    n_windows=None,
    test_size=len(test_df),
    step_size=1,
    refit=False,
)

print(f"Test predictions: {len(cv_test):,} rows")
print(f"Columns: {list(cv_test.columns)}")

In [ ]:
y_true = cv_test['y'].values
y_pred = cv_test['DeepAR'].values
lo_80  = cv_test['DeepAR-lo-80'].values
hi_80  = cv_test['DeepAR-hi-80'].values
lo_90  = cv_test['DeepAR-lo-90'].values
hi_90  = cv_test['DeepAR-hi-90'].values

results = compute_all(y_true, y_pred, lo_80, hi_80, lo_90, hi_90)

print("=== DeepAR Test Results ===")
print(f"  RMSE              : {results['rmse']:.6f}")
print(f"  MAE               : {results['mae']:.6f}")
print(f"  Directional Acc   : {results['dir_acc']:.4f}")
print(f"  Coverage 80%      : {results['coverage_80']:.4f}  (target: 0.80)")
print(f"  Coverage 90%      : {results['coverage_90']:.4f}  (target: 0.90)")
print(f"  Sharpe Ratio      : {results['sharpe']:.4f}")
print(f"  Max Drawdown      : {results['max_drawdown']:.6f}")

In [ ]:
n = 200
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(n), y_true[:n], label='Actual return_1h', alpha=0.8, linewidth=0.8, color='black')
ax.plot(range(n), y_pred[:n], label='DeepAR (mean)',    alpha=0.8, linewidth=0.8, color='steelblue')
ax.fill_between(range(n), lo_90[:n], hi_90[:n], alpha=0.12, color='steelblue', label='90% interval')
ax.fill_between(range(n), lo_80[:n], hi_80[:n], alpha=0.22, color='steelblue', label='80% interval')
ax.axhline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=9)
ax.set_title('DeepAR: predicted vs actual return_1h — first 200 test bars (2024)')
ax.set_xlabel('Test bar index')
ax.set_ylabel('return_1h')
plt.tight_layout()
plt.show()

In [ ]:
test_raw = pd.read_parquet('../data/splits/test.parquet').reset_index(drop=True)

preds_df = pd.DataFrame({
    'ds':       cv_test['ds'].values,
    'datetime': test_raw['datetime'].values[:len(cv_test)],
    'y':        y_true,
    'pred':     y_pred,
    'lo_80':    lo_80,
    'hi_80':    hi_80,
    'lo_90':    lo_90,
    'hi_90':    hi_90,
    'model':    'DeepAR',
})
preds_df.to_parquet('../data/predictions/deepar_preds.parquet', index=False)
print(f"Saved {len(preds_df):,} rows → data/predictions/deepar_preds.parquet")
print(preds_df.head(3))